# Evaluation Methodology — Report Section
> Author: Zachary Powell (zp21@rice.edu) — Evaluation Methods

---

## V. Evaluation Methodology

To rigorously measure and compare the performance of the three RAG architectures — Baseline RAG, Enhanced (LightRAG), and Agentic RAG (Hierarchical Retrieval) — we employ five evaluation metrics drawn from two traditional question-answering measures, one semantic similarity measure, and two modern RAG-specific benchmarks [1], [8]–[10]. All metrics are computed on the held-out test split of the HotpotQA dataset (7,405 examples, stratified by question type), which is carved from the official training partition so that the official validation set is reserved for hyperparameter selection during the fine-tuning stage.

### A. Traditional Metrics: Exact Match and Token-Level F1

**Exact Match (EM)** is a binary metric that returns 1 if the normalized model prediction matches the normalized ground-truth answer character-for-character, and 0 otherwise [1]. **Token-Level F1** relaxes this strict criterion by computing the harmonic mean of precision and recall at the token level, rewarding partial overlaps between the predicted and reference answer spans [1].

Both metrics apply the following normalization pipeline before comparison, following the official HotpotQA evaluation script [1]:

1. Lowercase the text.
2. Strip all punctuation characters.
3. Remove English articles (*a*, *an*, *the*).
4. Collapse all whitespace to a single space.

Formally, let $\hat{a}$ be the normalized predicted answer and $a^*$ be the normalized ground-truth answer with token sets $\hat{T}$ and $T^*$ respectively. Then:

$$\text{EM} = \mathbf{1}[\hat{a} = a^*]$$

$$\text{F1} = \frac{2 \cdot |\hat{T} \cap T^*|}{|\hat{T}| + |T^*|}$$

Corpus-level EM and F1 are reported as the mean over all test examples. These metrics serve as the primary lexical-correctness baselines for comparing the three architectures.

### B. Semantic Similarity Metric: BERTScore

BERTScore [8] addresses the limitation of exact-match metrics by evaluating semantic similarity through contextual token embeddings from a pretrained transformer. For each token in the predicted answer, BERTScore computes the maximum cosine similarity against all tokens in the reference answer using the encoder representations, and vice versa. Precision ($P_\text{BERT}$), Recall ($R_\text{BERT}$), and F1 ($F_\text{BERT}$) are then derived from these greedy-matched similarity scores and averaged over the corpus.

We use `roberta-large` as the scoring model, which is the standard BERTScore default and provides strong correlation with human judgments [8]. The primary reported figure is $F_\text{BERT}$. This metric is particularly valuable for HotpotQA because multi-hop answers are often phrased differently from the ground truth while remaining semantically correct (e.g., *"1984"* vs. *"the year 1984"*).

### C. Modern RAG-Specific Metrics

#### 1. RAGAS: Faithfulness and Answer Relevancy

RAGAS [9] provides automated evaluation of RAG pipelines by using an LLM judge to assess the relationship between the generated answer, the retrieved context, and the original question. We report two RAGAS metrics:

- **Faithfulness** measures the fraction of factual claims in the generated answer that are directly supported by the retrieved passages. A faithfulness score of 1.0 indicates that every claim in the answer is grounded in the retrieved evidence, while lower scores signal hallucination.
- **Answer Relevancy** measures how well the generated answer addresses the original question, regardless of factual correctness. It is computed by prompting the LLM to generate candidate questions from the answer and measuring their semantic similarity to the original question.

The LLM judge is `qwen2.5:7b` served via a local Ollama instance (no external API required). Because RAGAS relies on LLM inference for each example, evaluation is run on a random subsample of 200 test examples (with seed 42) to control computational cost while maintaining statistical reliability.

#### 2. RAGCap-Bench: Agentic Capability Evaluation

RAGCap-Bench [10] is a capability-oriented benchmark specifically designed for fine-grained evaluation of intermediate reasoning tasks in agentic RAG workflows. Rather than measuring end-to-end answer quality alone, it diagnoses specific model weaknesses across four core task dimensions via 255 curated multiple-choice questions drawn from diverse domains (entertainment, sports, technology, medicine):

| Task Type | Weight | Description |
|---|---|---|
| Planning | 29.8% | Decompose complex queries into sub-queries; formulate retrieval strategies |
| Evidence Extraction | 27.1% | Identify relevant information from retrieved documents; filter noise |
| Grounded Reasoning | 20.8% | Draw conclusions from retrieved evidence without unfounded inferences |
| Noise Robustness | 22.4% | Resist irrelevant, contradictory, or misleading retrieved passages |

We report per-dimension accuracy and a single **weighted accuracy** computed as:

$$\text{RAGCap}_\text{weighted} = 0.298 \cdot \text{Acc}_\text{plan} + 0.271 \cdot \text{Acc}_\text{evid} + 0.208 \cdot \text{Acc}_\text{reason} + 0.224 \cdot \text{Acc}_\text{noise}$$

RAGCap-Bench is particularly suited to differentiating the Agentic RAG architecture from the simpler Baseline and Enhanced systems, as planning and noise robustness demand iterative multi-step reasoning capabilities that single-pass retrieval systems cannot exhibit.

### D. Summary of Metrics

| Metric | Category | What it measures |
|---|---|---|
| Exact Match (EM) | Traditional | Strict lexical correctness |
| Token F1 | Traditional | Partial lexical overlap |
| BERTScore F1 | Semantic | Contextual semantic similarity |
| RAGAS Faithfulness | Modern RAG | Answer groundedness in retrieved context |
| RAGAS Answer Relevancy | Modern RAG | Relevance of answer to question |
| RAGCap-Bench Weighted Accuracy | Modern RAG | Agentic capability across 4 dimensions |

---
**References**  
[1] Z. Yang et al., "HotpotQA," arXiv:1809.09600, 2018.  
[8] T. Zhang et al., "BERTScore," arXiv:1904.09675, 2019.  
[9] J. James et al., "RAGAS," arXiv:2309.15217, 2025.  
[10] J. Lin et al., "RAGCap-Bench," arXiv:2510.13910, 2025.


# RAG Evaluation Pipeline
**COMP 652 – Natural Language Processing – Spring 2026 | Rice University**

**Author (Evaluation Section): Zachary Powell (zp21@rice.edu)**

---
### Metrics
| Category | Metric | Library / Backend |
|---|---|---|
| Traditional | Exact Match (EM) | custom (HotpotQA official logic) |
| Traditional | Token-level F1 | custom (HotpotQA official logic) |
| Semantic | BERTScore (P / R / F1) | `bert-score` |
| Modern RAG | Faithfulness, Answer Relevancy | `ragas` + Ollama `qwen2.5:7b` |
| Modern RAG | RAGCap-Bench capability probes | custom MCQ evaluator |

> **Generator model**: Fine-tuned Qwen (`qwen_hotpotqa_lora_final` LoRA adapters on `Qwen/Qwen3-4B-Instruct-2507`)  
> **RAGAS judge**: `qwen2.5:7b` via local Ollama server (no API key required)


## 0 · Dependency Installation

In [ ]:
import subprocess, sys

PACKAGES = [
    "datasets",
    "pandas",
    "scikit-learn",
    "bert-score",
    "ragas>=0.2",
    "langchain-community",  # Ollama backend for RAGAS (qwen2.5:7b via local Ollama server)
    "tqdm",
    "matplotlib",
    "seaborn",
]

subprocess.check_call([sys.executable, "-m", "pip", "install", "-U", "--quiet"] + PACKAGES)
print("All dependencies installed.")


CalledProcessError: Command '['b:\\Rice\\Comp652(Spring2026)\\.venv\\Scripts\\python.exe', '-m', 'pip', 'install', '-U', '--quiet', 'datasets', 'pandas', 'scikit-learn', 'bert-score', 'ragas>=0.2', 'langchain-openai', 'langchain-community', 'tqdm', 'matplotlib', 'seaborn']' returned non-zero exit status 1.

## 1 · Imports

In [ ]:
import re
import json
import string
import collections
import importlib
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

# BERTScore
from bert_score import score as bertscore_score

# RAGAS
from ragas import evaluate as ragas_evaluate
from ragas.metrics import faithfulness, answer_relevancy
from ragas.dataset_schema import SingleTurnSample, EvaluationDataset

print("Imports OK.")

## 2 · Load HotpotQA Data

In [ ]:
# --------------------------------------------------------------------------- #
# Inlined from hotpotqa_loader.ipynb (data loading + preprocessing helpers)   #
# --------------------------------------------------------------------------- #
from datasets import load_dataset
from sklearn.model_selection import train_test_split


def load_hotpotqa_mirror(config: str = "distractor") -> dict:
    """Load HotpotQA from archived raw JSON (script-free path)."""
    if config not in {"distractor", "fullwiki"}:
        raise ValueError("config must be 'distractor' or 'fullwiki'")
    url_base = "https://web.archive.org/web/20250512032701id_/http://curtis.ml.cmu.edu/datasets/hotpot/"
    data_files = {
        "train": url_base + "hotpot_train_v1.1.json",
        "validation": url_base + f"hotpot_dev_{config}_v1.json",
    }
    dataset = load_dataset("json", data_files=data_files)
    return {"train": dataset["train"], "validation": dataset["validation"]}


def flatten_context(example: dict) -> str:
    context = example["context"]
    if isinstance(context, dict):
        parts = [
            f"{t}: {' '.join(s)}"
            for t, s in zip(context.get("title", []), context.get("sentences", []))
        ]
        return " | ".join(parts)
    parts = []
    for item in context:
        if isinstance(item, (list, tuple)) and len(item) >= 2:
            sents = item[1] if isinstance(item[1], list) else [str(item[1])]
            parts.append(f"{item[0]}: {' '.join(sents)}")
    return " | ".join(parts)


def clean_text(text: str) -> str:
    return re.sub(r"\s+", " ", text.lower().strip())


def preprocess_split(dataset_split) -> pd.DataFrame:
    records = []
    for ex in dataset_split:
        supporting = ex["supporting_facts"]
        if isinstance(supporting, dict):
            s_titles = list(set(supporting.get("title", [])))
        else:
            s_titles = list({i[0] for i in supporting if isinstance(i, (list, tuple))})

        context = ex["context"]
        n_ctx = len(context.get("title", [])) if isinstance(context, dict) else len(context)
        records.append({
            "id": ex.get("id", ex.get("_id", "")),
            "question": clean_text(ex["question"]),
            "answer": clean_text(ex["answer"]),
            "question_type": ex["type"],
            "difficulty_level": ex["level"],
            "context_flat": flatten_context(ex),
            "supporting_titles": s_titles,
            "n_supporting_docs": len(s_titles),
            "n_context_docs": n_ctx,
            "answer_length": len(ex["answer"]),
        })
    return pd.DataFrame(records)


def create_splits(
    train_df: pd.DataFrame,
    val_df: pd.DataFrame,
    test_size: int = 7405,
    random_state: int = 42,
) -> dict[str, pd.DataFrame]:
    train_final, test_df = train_test_split(
        train_df, test_size=test_size,
        stratify=train_df["question_type"], random_state=random_state,
    )
    return {
        "train": train_final.reset_index(drop=True),
        "val":   val_df.reset_index(drop=True),
        "test":  test_df.reset_index(drop=True),
    }


# Load – comment out and set splits from a cached CSV if you have already run this.
raw    = load_hotpotqa_mirror(config="distractor")
splits = create_splits(
    preprocess_split(raw["train"]),
    preprocess_split(raw["validation"]),
    test_size=7405,
)

eval_df = splits["test"]   # Evaluation is always done on the held-out test split
print(f"Evaluation set: {len(eval_df):,} examples")
eval_df.head(3)

## 3 · Predictions Schema

### Standard format — Baseline & Enhanced RAG
Predictions files must be **JSON-Lines** (`.jsonl`) or CSV with these columns:

| Column | Type | Description |
|---|---|---|
| `id` | str | HotpotQA example id matching `eval_df["id"]` |
| `predicted_answer` | str | Model's generated answer |
| `retrieved_contexts` | list[str] | Retrieved passage strings fed to the generator |

`ground_truth_answer` and `question` are joined from `eval_df` automatically.

### Agentic RAG format — `predictions_agentic.jsonl` (provided ✓)
The agentic file uses a richer schema produced by the multi-step retrieval pipeline:

| Column | Type | Description |
|---|---|---|
| `qid` | str | HotpotQA example id |
| `pred_answer` | str | Final answer from the agentic pipeline |
| `trajectory` | list[dict] | Per-loop tool calls with `tool_name` and `tool_result` |
| `gold_answer` | str | Reference answer (used for cross-validation) |

`load_agentic_predictions()` adapts this to the standard schema automatically, extracting retrieved contexts from `read_chunk` tool call outputs in the trajectory.


In [ ]:
def load_predictions(path: str | Path, eval_df: pd.DataFrame) -> pd.DataFrame:
    """
    Load a predictions file (JSONL or CSV) and merge with the evaluation split.

    Required columns: id, predicted_answer
    Optional column : retrieved_contexts  (needed for RAGAS; BERTScore works without it)
    """
    p = Path(path)
    if p.suffix == ".jsonl":
        rows = [json.loads(line) for line in p.read_text(encoding="utf-8").splitlines() if line.strip()]
        preds = pd.DataFrame(rows)
    elif p.suffix == ".csv":
        preds = pd.read_csv(p)
    else:
        raise ValueError(f"Unsupported file type: {p.suffix}")

    # Normalise common alternate column names
    preds = preds.rename(columns={
        "qid": "id",
        "pred_answer": "predicted_answer",
        "retrieved_chunks": "retrieved_contexts",   # Chuan's baseline/enhanced files use this key
        "answer": "predicted_answer" if "predicted_answer" not in preds.columns else "answer",
    })

    required = {"id", "predicted_answer"}
    missing = required - set(preds.columns)
    if missing:
        raise ValueError(f"Predictions file is missing columns: {missing}")

    # Ensure retrieved_contexts exists — RAGAS needs it; other metrics do not
    if "retrieved_contexts" not in preds.columns:
        print("  [Note] 'retrieved_contexts' column not found — RAGAS will be skipped for this system.")
        preds["retrieved_contexts"] = [[] for _ in range(len(preds))]
    else:
        if preds["retrieved_contexts"].dtype == object:
            preds["retrieved_contexts"] = preds["retrieved_contexts"].apply(
                lambda x: json.loads(x) if isinstance(x, str) and x.startswith("[") else
                          ([x] if isinstance(x, str) else x)
            )

    # Drop columns from preds that also exist in eval_df to avoid _x/_y suffix conflicts
    drop_cols = [c for c in ["question", "gold_answer", "mode"] if c in preds.columns]
    preds = preds.drop(columns=drop_cols)
    merged = eval_df[["id", "question", "answer"]].merge(preds, on="id", how="inner")
    merged = merged.rename(columns={"answer": "ground_truth_answer"})
    print(f"  Loaded {len(merged):,} predictions (matched from {len(preds):,})")
    return merged


def make_dummy_predictions(eval_df: pd.DataFrame, n: int = 200) -> pd.DataFrame:
    """Build a small dummy predictions frame for smoke-testing the pipeline."""
    sub = eval_df.head(n).copy()
    sub["predicted_answer"] = sub["answer"]
    sub["retrieved_contexts"] = sub["context_flat"].apply(lambda c: [c[:500]])
    return sub[["id", "question", "answer", "predicted_answer", "retrieved_contexts"]].rename(
        columns={"answer": "ground_truth_answer"}
    )


print("load_predictions() and make_dummy_predictions() defined.")


In [ ]:

def load_agentic_predictions(path: str | Path, eval_df: pd.DataFrame) -> pd.DataFrame:
    """
    Load the agentic-RAG predictions JSONL and convert to the standard evaluation schema.

    Agentic JSONL fields used:
        qid           -> id
        pred_answer   -> predicted_answer
        trajectory    -> retrieved_contexts (text from read_chunk steps)

    Args:
        path    : Path to predictions_agentic.jsonl.
        eval_df : Evaluation DataFrame for question / ground-truth join.

    Returns:
        Merged DataFrame with columns:
            id, question, ground_truth_answer, predicted_answer, retrieved_contexts
    """
    p = Path(path)
    raw_rows = [json.loads(line) for line in p.read_text(encoding="utf-8").splitlines() if line.strip()]

    records = []
    for row in raw_rows:
        # Extract retrieved text from read_chunk tool calls in the trajectory
        contexts: list[str] = []
        for step in row.get("trajectory", []):
            if step.get("tool_name") == "read_chunk":
                tool_result = step.get("tool_result", "")
                # Parse text blocks between the dashed separator lines
                blocks = re.findall(r"-{80}\n(.*?)\n={80}", tool_result, re.DOTALL)
                contexts.extend(b.strip() for b in blocks if b.strip())

        # Fall back to keyword_search snippets if no read_chunk results were found
        if not contexts:
            for step in row.get("trajectory", []):
                if step.get("tool_name") == "keyword_search":
                    contexts.append(step.get("tool_result", "")[:1500])
                    break

        records.append({
            "id": row["qid"],
            "predicted_answer": row.get("pred_answer", ""),
            "retrieved_contexts": contexts if contexts else [""],
        })

    preds = pd.DataFrame(records)
    # Drop columns from preds that also exist in eval_df to avoid _x/_y suffix conflicts
    drop_cols = [c for c in ["question", "gold_answer", "mode"] if c in preds.columns]
    preds = preds.drop(columns=drop_cols)
    merged = eval_df[["id", "question", "answer"]].merge(preds, on="id", how="inner")
    merged = merged.rename(columns={"answer": "ground_truth_answer"})
    print(f"  Loaded {len(merged):,} agentic predictions (matched from {len(preds):,} in file)")
    return merged


print("load_agentic_predictions() defined.")


## 4 · Traditional Metrics — Exact Match & Token-Level F1

Implemented following the **official HotpotQA evaluation script** logic \[Yang et al., 2018\].

**Normalization steps** (applied to both prediction and reference before comparison):
1. Lowercase
2. Remove punctuation
3. Remove English articles (*a*, *an*, *the*)
4. Collapse whitespace

In [ ]:
def _normalize_answer(text: str) -> str:
    """Lowercase, strip punctuation, remove articles, collapse whitespace."""
    text = text.lower()
    text = text.translate(str.maketrans("", "", string.punctuation))
    text = re.sub(r"\b(a|an|the)\b", " ", text)
    return re.sub(r"\s+", " ", text).strip()


def exact_match(prediction: str, ground_truth: str) -> int:
    """Return 1 if normalized prediction equals normalized ground truth."""
    return int(_normalize_answer(prediction) == _normalize_answer(ground_truth))


def token_f1(prediction: str, ground_truth: str) -> float:
    """
    Compute token-level F1 between the predicted and ground-truth answers.
    Follows the HotpotQA official evaluation script.
    """
    pred_tokens = _normalize_answer(prediction).split()
    gold_tokens = _normalize_answer(ground_truth).split()

    common = collections.Counter(pred_tokens) & collections.Counter(gold_tokens)
    n_common = sum(common.values())

    if n_common == 0:
        return 0.0

    precision = n_common / len(pred_tokens)
    recall    = n_common / len(gold_tokens)
    return (2 * precision * recall) / (precision + recall)


def compute_em_f1(df: pd.DataFrame) -> dict[str, float]:
    """
    Compute corpus-level EM and F1 from a predictions DataFrame.

    Args:
        df: DataFrame with columns 'predicted_answer' and 'ground_truth_answer'.

    Returns:
        dict with keys 'EM' and 'F1' (mean over all examples).
    """
    ems, f1s = [], []
    for _, row in df.iterrows():
        ems.append(exact_match(row["predicted_answer"], row["ground_truth_answer"]))
        f1s.append(token_f1(row["predicted_answer"], row["ground_truth_answer"]))
    return {"EM": float(np.mean(ems)), "F1": float(np.mean(f1s))}


# Quick sanity check
# NOTE: avoid articles in test strings — _normalize_answer strips them,
# so "the cat sat" → "cat sat" (2 tokens), changing expected F1 from 2/3 to 1/2.
assert exact_match("The Eiffel Tower", "the eiffel tower") == 1
assert exact_match("Paris", "Berlin") == 0
assert abs(token_f1("big cat sat", "big cat mat") - 2/3) < 1e-6
print("EM / F1 helpers verified.")


## 5 · Semantic Similarity — BERTScore

BERTScore computes token-level similarity between generated and reference answers using contextual
embeddings from a pretrained BERT-variant \[Zhang et al., 2019\].  We report **Precision, Recall,
and F1** (the primary reported figure).

Model used: `roberta-large` — the standard BERTScore default, with well-tested tokenizer
compatibility across platforms.  (`microsoft/deberta-xlarge-mnli` has a known `model_max_length`
overflow in its fast tokenizer on Python 3.13 / tokenizers ≥ 0.20 and is avoided.)


In [ ]:
def compute_bertscore(
    df: pd.DataFrame,
    model_type: str = "roberta-large",
    batch_size: int = 64,
    device: str = "cuda",
    verbose: bool = True,
) -> dict[str, float]:
    """
    Compute corpus-level BERTScore P / R / F1.

    Uses BERTScorer directly (instead of bert_score.score) so we can patch
    tokenizer.model_max_length before scoring.  Newer transformers (>=4.44) sets
    model_max_length to 1e30 for fast tokenizers that lack an explicit config value;
    the Rust tokenizer backend overflows on any value > sys.maxsize.

    Args:
        df          : DataFrame with 'predicted_answer' and 'ground_truth_answer'.
        model_type  : Hugging Face model identifier for BERTScore.
        batch_size  : Mini-batch size for encoding.
        device      : 'cuda' or 'cpu'.
        verbose     : Show a progress bar from bert_score.

    Returns:
        dict with keys 'BERTScore_P', 'BERTScore_R', 'BERTScore_F1'.
    """
    from bert_score import BERTScorer

    predictions   = df["predicted_answer"].tolist()
    ground_truths = df["ground_truth_answer"].tolist()

    scorer = BERTScorer(model_type=model_type, device=device, lang="en")

    # ── Patch: cap tokenizer.model_max_length to avoid Rust overflow ──────── #
    # transformers >= 4.44 sets model_max_length = int(1e30) when not in config.
    # bert_score passes this directly to tokenizer.encode(..., max_length=...).
    # The Rust tokenizer backend cannot handle values > 2^31 on 32-bit or ~2^63.
    if scorer._tokenizer.model_max_length > 1_000_000:
        scorer._tokenizer.model_max_length = 512

    P, R, F1 = scorer.score(
        predictions,
        ground_truths,
        verbose=verbose,
        batch_size=batch_size,
    )

    return {
        "BERTScore_P":  float(P.mean()),
        "BERTScore_R":  float(R.mean()),
        "BERTScore_F1": float(F1.mean()),
    }


print("compute_bertscore() defined.")


## 6 · Modern RAG Metrics — RAGAS

RAGAS provides automated evaluation of RAG pipelines by measuring whether generated answers are
grounded in the retrieved evidence \[James et al., 2025\].  We report two primary metrics:

| Metric | What it measures |
|---|---|
| **Faithfulness** | Fraction of answer claims that are supported by the retrieved context |
| **Answer Relevancy** | Semantic relevance of the answer to the question |

RAGAS uses an LLM judge internally.  We use **`qwen2.5:7b` via Ollama** (already running on this
machine — no API key required).  To switch models, change the `model=` argument in the cell below.


In [ ]:
import os

# ---- LLM backend: Ollama (running locally on this machine) --------------- #
# Uses qwen2.5:7b via the Ollama server (no API key needed).                  #
# Embeddings: nomic-embed-text via Ollama (avoids OpenAI fallback in RAGAS).  #
# To switch models: change model= below, e.g. "qwen2.5:14b"                  #
# ------------------------------------------------------------------------- #
ragas_llm       = None
ragas_embeddings = None

try:
    from langchain_community.llms import Ollama
    from langchain_community.embeddings import OllamaEmbeddings
    from ragas.llms import LangchainLLMWrapper
    from ragas.embeddings import LangchainEmbeddingsWrapper

    _ollama_llm = Ollama(model="qwen2.5:7b")
    _ = _ollama_llm.invoke("Say OK")                    # connectivity check
    ragas_llm = LangchainLLMWrapper(_ollama_llm)

    _ollama_emb = OllamaEmbeddings(model="nomic-embed-text")
    _ = _ollama_emb.embed_query("test")                 # connectivity check
    ragas_embeddings = LangchainEmbeddingsWrapper(_ollama_emb)

    print("RAGAS LLM backend:        Ollama qwen2.5:7b")
    print("RAGAS Embeddings backend: Ollama nomic-embed-text")
except Exception as e:
    ragas_llm = None
    ragas_embeddings = None
    print(f"[Warning] RAGAS backend not available: {e}")


In [ ]:
def compute_ragas(
    df: pd.DataFrame,
    llm=None,
    embeddings=None,
    sample_size: int | None = 200,
    random_state: int = 42,
) -> dict[str, float]:
    """
    Compute RAGAS Faithfulness and Answer Relevancy.

    Skips gracefully if the LLM backend is not configured or if retrieved_contexts
    are missing (all empty lists).

    Args:
        df           : DataFrame with predictions.
        llm          : LangchainLLMWrapper around an Ollama/OpenAI LLM.
        embeddings   : LangchainEmbeddingsWrapper for answer_relevancy metric.
                       Must be provided to avoid RAGAS falling back to OpenAI.
        sample_size  : Max examples to evaluate (None = full set).
        random_state : Random seed for sampling.
    """
    if llm is None:
        print("[Skip] RAGAS requires an LLM backend. Configure ragas_llm above.")
        return {"RAGAS_Faithfulness": float("nan"), "RAGAS_AnswerRelevancy": float("nan")}

    # Skip if no retrieved contexts were saved alongside predictions
    has_contexts = df["retrieved_contexts"].apply(lambda c: len(c) > 0).any()
    if not has_contexts:
        print("[Skip] RAGAS — no retrieved_contexts available for this system.")
        return {"RAGAS_Faithfulness": float("nan"), "RAGAS_AnswerRelevancy": float("nan")}

    work_df = df.copy()
    if sample_size is not None and sample_size < len(work_df):
        work_df = work_df.sample(n=sample_size, random_state=random_state).reset_index(drop=True)
        print(f"  Evaluating RAGAS on {sample_size} sampled examples …")

    samples = [
        SingleTurnSample(
            user_input=row["question"],
            response=row["predicted_answer"],
            reference=row["ground_truth_answer"],
            retrieved_contexts=row["retrieved_contexts"] if row["retrieved_contexts"] else [""],
        )
        for _, row in work_df.iterrows()
    ]

    dataset = EvaluationDataset(samples=samples)

    # Create fresh metric instances and wire llm/embeddings directly.
    # Prevents RAGAS from falling back to OpenAI when provider inference
    # cannot recognise LangchainLLMWrapper(Ollama(...)).
    from ragas.metrics import Faithfulness, AnswerRelevancy
    _faithfulness = Faithfulness()
    _faithfulness.llm = llm
    _answer_relevancy = AnswerRelevancy()
    _answer_relevancy.llm = llm
    metrics_to_run = [_faithfulness]
    if embeddings is not None:
        _answer_relevancy.embeddings = embeddings
        metrics_to_run.append(_answer_relevancy)
    else:
        print("[Note] No embeddings available — skipping AnswerRelevancy.")

    try:
        results = ragas_evaluate(
            dataset=dataset,
            metrics=metrics_to_run,
        )
        scores_df = results.to_pandas()
        faith_score = float(scores_df["faithfulness"].mean()) if "faithfulness" in scores_df.columns else float("nan")
        ar_score    = float(scores_df["answer_relevancy"].mean()) if "answer_relevancy" in scores_df.columns else float("nan")
        return {
            "RAGAS_Faithfulness":    faith_score,
            "RAGAS_AnswerRelevancy": ar_score,
        }
    except Exception as e:
        print(f"[Warning] RAGAS evaluation failed: {e}")
        return {"RAGAS_Faithfulness": float("nan"), "RAGAS_AnswerRelevancy": float("nan")}


print("compute_ragas() defined.")


## 7 · Modern RAG Metrics — RAGCap-Bench

RAGCap-Bench \[Lin et al., 2025\] is a capability-oriented benchmark of **255 curated MCQ** that
diagnoses four agentic RAG capability dimensions:

| Task type | Weight | What it tests |
|---|---|---|
| Planning | 29.8 % | Decompose queries → sub-queries; formulate retrieval strategies |
| Evidence Extraction | 27.1 % | Identify relevant information; filter noise |
| Grounded Reasoning | 20.8 % | Draw conclusions from retrieved evidence only |
| Noise Robustness | 22.4 % | Resist irrelevant / contradictory retrieved passages |



In [ ]:
# ---- Configuration -------------------------------------------------------- #
RAGCAP_PATH: str | None = None
# --------------------------------------------------------------------------- #


def load_ragcap_bench(path: str) -> pd.DataFrame:
    """
    Load the RAGCap-Bench MCQ dataset from a local JSON file.

    Expected JSON schema per item:
        {
          "id": str,
          "question": str,
          "choices": ["A) ...", "B) ...", "C) ...", "D) ..."],
          "answer": "A" | "B" | "C" | "D",
          "task_type": "planning" | "evidence_extraction" | "grounded_reasoning" | "noise_robustness",
          "context": str    # retrieved passages provided with the question
        }
    """
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    df = pd.DataFrame(data)
    print(f"  Loaded RAGCap-Bench: {len(df)} questions")
    print(df["task_type"].value_counts().to_string())
    return df


def evaluate_ragcap(
    model_fn,
    ragcap_df: pd.DataFrame,
    batch_size: int = 16,
) -> dict[str, float]:
    """
    Evaluate a model on the RAGCap-Bench MCQ questions.

    Args:
        model_fn    : Callable(question: str, context: str, choices: list[str]) -> str
                      Returns the selected option letter ("A", "B", "C", or "D").
        ragcap_df   : DataFrame loaded by load_ragcap_bench().
        batch_size  : Number of questions per batch (for progress reporting).

    Returns:
        dict with per-task-type accuracy and an overall weighted accuracy.
    """
    TASK_WEIGHTS = {
        "planning":             0.298,
        "evidence_extraction":  0.271,
        "grounded_reasoning":   0.208,
        "noise_robustness":     0.224,
    }

    results_per_task: dict[str, list[int]] = {t: [] for t in TASK_WEIGHTS}

    for _, row in tqdm(ragcap_df.iterrows(), total=len(ragcap_df), desc="RAGCap-Bench"):
        predicted = model_fn(
            question=row["question"],
            context=row["context"],
            choices=row["choices"],
        )
        correct = int(predicted.strip().upper() == row["answer"].strip().upper())
        task = row["task_type"]
        if task in results_per_task:
            results_per_task[task].append(correct)

    scores: dict[str, float] = {}
    weighted_sum = 0.0
    for task, weight in TASK_WEIGHTS.items():
        acc = float(np.mean(results_per_task[task])) if results_per_task[task] else float("nan")
        scores[f"RAGCap_{task.replace(' ', '_')}"] = acc
        if not np.isnan(acc):
            weighted_sum += weight * acc
    scores["RAGCap_WeightedAccuracy"] = weighted_sum

    return scores


print("RAGCap-Bench helpers defined.")
if RAGCAP_PATH is None:
    print("[Note] Set RAGCAP_PATH to the local benchmark JSON to enable full RAGCap evaluation.")

## 8 · Unified Evaluation Runner

Call `run_full_evaluation()` once per RAG system to get all five metrics in a single dict.

In [ ]:
def run_full_evaluation(
    system_name: str,
    predictions_df: pd.DataFrame,
    bertscore_model: str = "roberta-large",
    bertscore_device: str = "cuda",
    ragas_llm=None,
    ragas_embeddings=None,
    ragas_sample: int = 200,
    ragcap_df: pd.DataFrame | None = None,
    ragcap_model_fn=None,
) -> dict[str, Any]:
    """
    Run the full evaluation suite for one RAG system.

    Args:
        system_name       : Human-readable label (e.g. 'Baseline RAG').
        predictions_df    : Output of load_predictions() for this system.
        bertscore_model   : HuggingFace model id for BERTScore.
                            roberta-large is the default; deberta-xlarge-mnli has a
                            known tokenizer overflow on Python 3.13 / tokenizers >= 0.20.
        bertscore_device  : 'cuda' or 'cpu'.
        ragas_llm         : LangchainLLMWrapper for RAGAS (None = skip).
        ragas_embeddings  : LangchainEmbeddingsWrapper for RAGAS answer_relevancy.
                            Must be provided to prevent RAGAS falling back to OpenAI.
        ragas_sample      : Number of examples for RAGAS (None = full set).
        ragcap_df         : RAGCap-Bench DataFrame (None = skip).
        ragcap_model_fn   : Callable for RAGCap MCQ evaluation (None = skip).

    Returns:
        dict with keys: system, EM, F1, BERTScore_P, BERTScore_R, BERTScore_F1,
                        RAGAS_Faithfulness, RAGAS_AnswerRelevancy,
                        RAGCap_* (if available)
    """
    print(f"\n{'='*60}")
    print(f"  Evaluating: {system_name}  ({len(predictions_df):,} examples)")
    print(f"{'='*60}")

    scores: dict[str, Any] = {"system": system_name}

    # ── (1) Exact Match & F1 ──────────────────────────────────────────────── #
    print("[1/4] Computing EM and Token F1 …")
    scores.update(compute_em_f1(predictions_df))

    # ── (2) BERTScore ─────────────────────────────────────────────────────── #
    print("[2/4] Computing BERTScore …")
    scores.update(
        compute_bertscore(predictions_df, model_type=bertscore_model, device=bertscore_device)
    )

    # ── (3) RAGAS ─────────────────────────────────────────────────────────── #
    print("[3/4] Computing RAGAS …")
    scores.update(compute_ragas(
        predictions_df,
        llm=ragas_llm,
        embeddings=ragas_embeddings,
        sample_size=ragas_sample,
    ))

    # ── (4) RAGCap-Bench ──────────────────────────────────────────────────── #
    if ragcap_df is not None and ragcap_model_fn is not None:
        print("[4/4] Computing RAGCap-Bench …")
        scores.update(evaluate_ragcap(ragcap_model_fn, ragcap_df))
    else:
        print("[4/4] RAGCap-Bench skipped (set ragcap_df and ragcap_model_fn).")

    print(f"\n  Results for {system_name}:")
    for k, v in scores.items():
        if k != "system":
            print(f"    {k:30s}: {v:.4f}" if isinstance(v, float) else f"    {k}: {v}")

    return scores


print("run_full_evaluation() defined.")


## 9 · Smoke Test

In [ ]:
dummy_preds = make_dummy_predictions(eval_df, n=200)

smoke_results = run_full_evaluation(
    system_name="Smoke Test (perfect)",
    predictions_df=dummy_preds,
    bertscore_model="roberta-large",
    bertscore_device="cpu",
    ragas_llm=ragas_llm,
    ragas_sample=5,
)


## 10 · Evaluate All Three RAG Architectures

In [ ]:

# ── Prediction file paths ──────────────────────────────────────────────────────────── #
BASELINE_PREDS_PATH = Path("outputs/predictions_baseline.jsonl")  # n=150 (Chuan's repo ✓)
ENHANCED_PREDS_PATH = Path("outputs/predictions_enhanced.jsonl")  # n=150 (Chuan's repo ✓)
AGENTIC_PREDS_PATH  = Path("predictions_agentic.jsonl")           # n=50  (local ✓)
# ──────────────────────────────────────────────────────────────────────────────────── #

import torch
_device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {_device}  |  BERTScore model: roberta-large")

all_results = []

EVAL_KWARGS = dict(
    bertscore_model="roberta-large",  # deberta-xlarge-mnli has tokenizer overflow on Python 3.13
    bertscore_device=_device,
    ragas_llm=ragas_llm,
    ragas_embeddings=ragas_embeddings,
    ragas_sample=30,
)

# ── Baseline RAG ──────────────────────────────────────────────────────────────────── #
if BASELINE_PREDS_PATH.exists():
    baseline_preds = load_predictions(BASELINE_PREDS_PATH, eval_df)
    all_results.append(run_full_evaluation(
        system_name="Baseline RAG",
        predictions_df=baseline_preds,
        **EVAL_KWARGS,
        ragcap_df=None,
        ragcap_model_fn=None,
    ))
else:
    print(f"[Skip] Baseline RAG — file not found: {BASELINE_PREDS_PATH}")

# ── Enhanced RAG ──────────────────────────────────────────────────────────────────── #
if ENHANCED_PREDS_PATH.exists():
    enhanced_preds = load_predictions(ENHANCED_PREDS_PATH, eval_df)
    all_results.append(run_full_evaluation(
        system_name="Enhanced RAG",
        predictions_df=enhanced_preds,
        **EVAL_KWARGS,
        ragcap_df=None,
        ragcap_model_fn=None,
    ))
else:
    print(f"[Skip] Enhanced RAG — file not found: {ENHANCED_PREDS_PATH}")

# ── Agentic RAG ───────────────────────────────────────────────────────────────────── #
if AGENTIC_PREDS_PATH.exists():
    agentic_preds = load_agentic_predictions(AGENTIC_PREDS_PATH, eval_df)
    all_results.append(run_full_evaluation(
        system_name="Agentic RAG",
        predictions_df=agentic_preds,
        **EVAL_KWARGS,
        ragcap_df=None,
        ragcap_model_fn=None,
    ))
else:
    print(f"[Skip] Agentic RAG — file not found: {AGENTIC_PREDS_PATH}")

print(f"\nFinished evaluating {len(all_results)} system(s).")


## 11 · Results Table and Visualisations

In [ ]:
if not all_results:
    # Fall back to smoke test results so the notebook renders cleanly
    all_results = [smoke_results]

results_df = pd.DataFrame(all_results).set_index("system")

# Format for display
display_df = results_df.copy()
for col in display_df.columns:
    display_df[col] = display_df[col].apply(
        lambda v: f"{v:.4f}" if isinstance(v, float) and not np.isnan(v) else v
    )

print("\n=== Evaluation Results Summary ===")
print(display_df.to_string())

In [ ]:
# ── Bar chart: core metrics by architecture ──────────────────────────────── #
PLOT_METRICS = ["EM", "F1", "BERTScore_F1", "RAGAS_Faithfulness", "RAGAS_AnswerRelevancy"]

plot_df = results_df[[c for c in PLOT_METRICS if c in results_df.columns]].copy().astype(float)

fig, axes = plt.subplots(1, len(plot_df.columns), figsize=(4 * len(plot_df.columns), 5), sharey=False)
if len(plot_df.columns) == 1:
    axes = [axes]

colors = sns.color_palette("Set2", n_colors=len(plot_df))

for ax, metric in zip(axes, plot_df.columns):
    bars = ax.bar(plot_df.index, plot_df[metric], color=colors, edgecolor="black", linewidth=0.7)
    ax.set_title(metric, fontsize=11, fontweight="bold")
    ax.set_ylim(0, 1.05)
    ax.set_ylabel("Score", fontsize=9)
    ax.tick_params(axis="x", rotation=20, labelsize=8)
    for bar, v in zip(bars, plot_df[metric]):
        if not np.isnan(v):
            ax.text(bar.get_x() + bar.get_width() / 2, v + 0.02, f"{v:.3f}",
                    ha="center", va="bottom", fontsize=8)

fig.suptitle("RAG Architecture Comparison — HotpotQA Evaluation", fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig("rag_evaluation_results.pdf", bbox_inches="tight")
plt.show()
print("Figure saved as rag_evaluation_results.pdf")

In [ ]:
# ── Heatmap: all metrics ─────────────────────────────────────────────────── #
numeric_df = results_df.select_dtypes(include="number").astype(float)

fig, ax = plt.subplots(figsize=(max(8, len(numeric_df.columns) * 1.2), max(3, len(numeric_df) * 0.9)))
sns.heatmap(
    numeric_df,
    annot=True,
    fmt=".3f",
    cmap="YlGnBu",
    linewidths=0.5,
    ax=ax,
    vmin=0,
    vmax=1,
)
ax.set_title("All Metrics — RAG Architecture Comparison", fontsize=12, fontweight="bold")
ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha="right", fontsize=9)
plt.tight_layout()
plt.savefig("rag_evaluation_heatmap.pdf", bbox_inches="tight")
plt.show()
print("Heatmap saved as rag_evaluation_heatmap.pdf")

In [ ]:
results_df.to_csv("rag_evaluation_results.csv")
print("Results saved to rag_evaluation_results.csv")
results_df